# A live Veusz figure in a Jupyter notebook

This notebook builds a **phase diagram** and renders it with a live, interactive
**`VeuszWidget`** — a Veusz figure that runs *inside this kernel* (it's an
[anywidget](https://anywidget.dev), so it works unchanged in JupyterLab, classic
Notebook, **VS Code**, JupyterLite, and Colab). Drag a rectangle on the figure to
zoom; double-click to reset. Click **⚙ Edit** under the figure for a full GUI:
change any setting, pick colormaps from a searchable swatch list, and use the
toolbar to **Insert** pages / graphs / axes / plotters (XY, function, bar, fit,
image, contour, …) plus duplicate / delete / move / rename / undo / redo.
Multi-page documents get **page tabs** above the figure.

The figure renders to vector SVG with a bundled WebAssembly backend — no Qt, no
server, no network. The notebook's arrays stay in the kernel; only the finished
scene crosses to the frontend.

In [ ]:
# Setup. In JupyterLite/Pyodide, install the stack with micropip; in a local
# Jupyter / VS Code kernel these are already pip-installed, so this is a no-op.
try:
    import micropip  # noqa: F401  (present only under Pyodide / JupyterLite)
    await micropip.install([
        "anywidget", "fonttools",
        "https://yipihey.github.io/veusz/embed/v4.5.0/veusz-4.5.0-py3-none-any.whl",
    ])
except ModuleNotFoundError:
    pass


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
n = 6000
logrho = np.concatenate([rng.normal(-1.0, 0.7, n), rng.normal(1.2, 0.4, n)])
logT = np.concatenate([4.2 + 0.55*rng.normal(-1.0, 0.7, n) + rng.normal(0, 0.2, n),
                       6.4 + rng.normal(0, 0.3, n)])
print(f"{logrho.size:,} cells")


In [ ]:
from veusz.notebook import VeuszWidget

# A compact Veusz `density` 2D-histogram document. Data is fed from the kernel
# below (set_data), so the figure binds to THIS notebook's arrays.
PHASE = r"""SetCompatLevel(0)
Add('page', name='page1', autoadd=False)
To('page1')
Add('graph', name='graph1', autoadd=False)
To('graph1')
Add('axis', name='x', autoadd=False)
To('x')
Set('label', 'log \\rho')
To('..')
Add('axis', name='y', autoadd=False)
To('y')
Set('label', 'log T')
Set('direction', 'vertical')
To('..')
Add('density', name='dens', autoadd=False)
To('dens')
Set('xData', 'logrho')
Set('yData', 'logT')
Set('numBinsX', 120)
Set('numBinsY', 120)
Set('colorMap', 'viridis')
Set('colorScaling', 'log')
To('..')
Add('colorbar', name='colorbar1', autoadd=False)
To('colorbar1')
Set('widgetName', 'dens')
Set('label', 'counts')
To('..')
To('..')
To('..')
"""

fig = VeuszWidget(vsz=PHASE, width=640, height=560)
fig.set_data("logrho", logrho)
fig.set_data("logT", logT)
fig


You can edit the figure two ways:

* **GUI** — click **⚙ Edit** beneath the figure. Choose a widget in the
  dropdown and change its settings with the controls (colormaps get a
  searchable swatch picker). The toolbar above adds the full Veusz stack —
  **Insert** new pages, graphs, axes, plotters (XY, function, bar, fit,
  image, contour, vector field…), keys and labels — plus duplicate, delete,
  move, rename, and undo/redo. Every edit applies live.
* **Code** — set any property by path and it redraws in place:

```python
fig.set_setting("/page1/graph1/dens/colorMap", "plasma")
```